# Caso 07 — Ampara, seguros: ¿qué canal paga lo que cuesta?
## Día 2 — Limpiar, y decidir tú qué es el margen

Con el mapa del Día 1 en la cabeza, hoy limpiamos los 7 ficheros de forma ordenada, comprobando cada paso, y dejamos escrita la definición operativa de la métrica central de la semana: el **margen técnico de un canal**.

### Plan de Limpieza y Decisiones de Criterio:
1. **Fechas en 3 formatos** (`YYYY-MM-DD`, `DD/MM/YYYY` y mes en inglés `12 Mar 2024`): unificar con `pd.to_datetime(col, format='mixed', dayfirst=True)` y verificar que quedan `0` `NaT` inesperados (conservando el vacío con significado en `siniestros.fecha_cierre`).
2. **Importes en euros y porcentajes en texto**:
   - Función reutilizable `limpiar_euros`: quitar `€` y espacios $ightarrow$ quitar `.` de miles $ightarrow$ cambiar `,` decimal por `.` $ightarrow$ convertir con `pd.to_numeric(..., errors='coerce')`.
   - Limpiar `descuento_pct` quitando `%` y espacios, rellenando los vacíos con `0` (sin descuento).
3. **Deduplicación y filtrado de valores imposibles / outliers**:
   - Deduplicar `polizas` por `poliza_id` (`-600` duplicados de webhook) y `siniestros` por `siniestro_id` (`-857` duplicados por reapertura).
   - Filtrar `prima_anual > 0` en `polizas` (excluye las `15` primas cero/negativas basura) e `importe > 0` e `importe < 1_000_000` en `siniestros` (excluye negativos por recobro y el outlier `9.999.999 €`).
4. **Normalización de dimensiones críticas (0 nulos nuevos)**:
   - Mapear las **22 grafías** de `polizas.canal` y las **12 grafías** de `campanas.canal` a los **5 canales canónicos**: `comparadores`, `directo`, `mediadores`, `banca`, `referido`.
   - Extraer el **ramo real** (`auto`, `hogar`, `salud`, `vida`, `mascotas`) a partir de `productos.nombre`.
   - Normalizar `polizas.tipo` (`nueva`, `renovacion`), `polizas.estado` (`vigente`, `vencida`, `anulada`), `siniestros.estado` (`cerrado`, `abierto`, `rechazado`) y `siniestros.fraude` (booleano explícito `True`/`False`).
   - Normalizar `clientes.provincia` (dejando los vacíos reales como `'desconocido'`), `clientes.sexo` y `mediadores`.
5. **Definición del Margen Técnico de un Canal** y primer recuento limpio de pólizas nuevas por canal.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("data/raw")
CLEAN_DIR = Path("data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

canales = pd.read_csv(RAW_DIR / "canales.csv")
productos = pd.read_csv(RAW_DIR / "productos.csv")
mediadores = pd.read_csv(RAW_DIR / "mediadores.csv")
campanas = pd.read_csv(RAW_DIR / "campanas.csv")
clientes = pd.read_csv(RAW_DIR / "clientes.csv")
polizas = pd.read_csv(RAW_DIR / "polizas.csv")
siniestros = pd.read_csv(RAW_DIR / "siniestros.csv")

print("Archivos crudos cargados correctamente.")


Archivos crudos cargados correctamente.


### 1. Limpieza de Fechas (`format='mixed', dayfirst=True`) y Funciones Monetarias
Aplicamos el orden seguro para euros (`quitar € y espacios -> quitar punto de miles -> cambiar coma por punto -> float`) y parseamos todas las columnas de fecha.


In [2]:
def limpiar_euros(serie: pd.Series) -> pd.Series:
    """Convierte cadenas en formato español ('1.250,00 €') a float respetando el orden seguro."""
    limpia = (
        serie.astype(str)
        .str.replace("€", "", regex=False)
        .str.replace("\ufffd", "", regex=False)
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(limpia, errors="coerce")

def limpiar_porcentaje(serie: pd.Series) -> pd.Series:
    """Convierte porcentajes en texto ('15 %') a número entero/float; vacíos pasan a 0."""
    limpia = (
        serie.fillna("0")
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(limpia, errors="coerce").fillna(0.0)

def parsear_fecha(serie: pd.Series) -> pd.Series:
    return pd.to_datetime(serie, format="mixed", dayfirst=True, errors="coerce")

# Aplicamos a productos, campanas, clientes, polizas y siniestros
productos["franquicia"] = limpiar_euros(productos["franquicia"])

campanas["coste"] = limpiar_euros(campanas["coste"])
campanas["fecha_inicio"] = parsear_fecha(campanas["fecha_inicio"])
campanas["fecha_fin"] = parsear_fecha(campanas["fecha_fin"])

clientes["fecha_nacimiento"] = parsear_fecha(clientes["fecha_nacimiento"])
clientes["fecha_alta"] = parsear_fecha(clientes["fecha_alta"])

polizas["prima_anual"] = limpiar_euros(polizas["prima_anual"])
polizas["descuento_pct"] = limpiar_porcentaje(polizas["descuento_pct"])
polizas["fecha_efecto"] = parsear_fecha(polizas["fecha_efecto"])
polizas["fecha_vencimiento"] = parsear_fecha(polizas["fecha_vencimiento"])

siniestros["importe"] = limpiar_euros(siniestros["importe"])
siniestros["fecha_apertura"] = parsear_fecha(siniestros["fecha_apertura"])
siniestros["fecha_cierre"] = parsear_fecha(siniestros["fecha_cierre"])

print("Comprobación de NaT tras parsear fechas:")
print(f"- campanas.fecha_inicio NaT: {campanas['fecha_inicio'].isna().sum()} | fecha_fin NaT: {campanas['fecha_fin'].isna().sum()}")
print(f"- clientes.fecha_nacimiento NaT: {clientes['fecha_nacimiento'].isna().sum()} | fecha_alta NaT: {clientes['fecha_alta'].isna().sum()}")
print(f"- polizas.fecha_efecto NaT: {polizas['fecha_efecto'].isna().sum()} | fecha_vencimiento NaT: {polizas['fecha_vencimiento'].isna().sum()}")
print(f"- siniestros.fecha_apertura NaT: {siniestros['fecha_apertura'].isna().sum()} | fecha_cierre NaT (abiertos conservados): {siniestros['fecha_cierre'].isna().sum():,}")


Comprobación de NaT tras parsear fechas:
- campanas.fecha_inicio NaT: 0 | fecha_fin NaT: 4
- clientes.fecha_nacimiento NaT: 0 | fecha_alta NaT: 0
- polizas.fecha_efecto NaT: 0 | fecha_vencimiento NaT: 0
- siniestros.fecha_apertura NaT: 0 | fecha_cierre NaT (abiertos conservados): 20,568


### 2. Normalización de Canales, Ramos, Estados y Fraude (Comprobando 0 Nulos Nuevos)
Mapeamos las 22 grafías de `polizas.canal` y las 12 de `campanas.canal` a los 5 canales canónicos (`comparadores`, `directo`, `mediadores`, `banca`, `referido`), y extraemos el ramo real de `productos.nombre`.


In [3]:
# 1) Mapa de canal para polizas (22 grafías -> 5 canónicos)
mapa_canal_polizas = {
    "comparadores": "comparadores",
    "comparador": "comparadores",
    "rastreator": "comparadores",
    "agregador": "comparadores",
    "directo": "directo",
    "web": "directo",
    "app": "directo",
    "online": "directo",
    "mediadores": "mediadores",
    "mediador": "mediadores",
    "corredor": "mediadores",
    "agente": "mediadores",
    "banca": "banca",
    "bancaseguros": "banca",
    "banco": "banca",
    "referido": "referido",
    "recomendacion": "referido",
    "recomendación": "referido",
    "amigo": "referido",
}

polizas["canal"] = polizas["canal"].astype(str).str.strip().str.lower().map(mapa_canal_polizas)
assert polizas["canal"].isna().sum() == 0, f"Quedaron nulos en polizas.canal: {polizas['canal'].isna().sum()}"

# 2) Mapa de canal para campanas (12 grafías -> 5 canónicos)
mapa_canal_campanas = {
    **mapa_canal_polizas,
    "ads": "directo",
    "paid": "directo",
    "comision": "mediadores",
    "comisión": "mediadores",
    "acuerdo": "banca",
    "recompensa": "referido",
}
campanas["canal"] = campanas["canal"].astype(str).str.strip().str.lower().map(mapa_canal_campanas)
assert campanas["canal"].isna().sum() == 0, f"Quedaron nulos en campanas.canal: {campanas['canal'].isna().sum()}"

# 3) Ramo real en productos (escondido en la columna 'nombre')
mapa_ramo_productos = {
    "automovil": "auto",
    "automóvil": "auto",
    "auto": "auto",
    "vivienda": "hogar",
    "hogar": "hogar",
    "salud": "salud",
    "vida": "vida",
    "mascota": "mascotas",
    "mascotas": "mascotas",
}
productos["ramo_agrupado"] = productos["ramo"].astype(str).str.strip().str.lower()
productos["ramo"] = productos["nombre"].astype(str).str.strip().str.lower().map(mapa_ramo_productos)
productos["nombre"] = productos["ramo"]
assert productos["ramo"].isna().sum() == 0, "Quedaron nulos en productos.ramo"

# 4) Tipo y estado en polizas
mapa_tipo_poliza = {
    "nueva": "nueva",
    "alta": "nueva",
    "renovacion": "renovacion",
    "renovación": "renovacion",
    "renov": "renovacion",
    "renovada": "renovacion",
}
polizas["tipo"] = polizas["tipo"].astype(str).str.strip().str.lower().map(mapa_tipo_poliza)
assert polizas["tipo"].isna().sum() == 0, "Quedaron nulos en polizas.tipo"

mapa_estado_poliza = {
    "vigente": "vigente",
    "activa": "vigente",
    "vencida": "vencida",
    "anulada": "anulada",
    "cancelada": "anulada",
    "baja": "anulada",
}
polizas["estado"] = polizas["estado"].astype(str).str.strip().str.lower().map(mapa_estado_poliza)
assert polizas["estado"].isna().sum() == 0, "Quedaron nulos en polizas.estado"

# 5) Estado y fraude en siniestros
mapa_estado_siniestro = {
    "pagado": "cerrado",
    "cerrado": "cerrado",
    "abierto": "abierto",
    "en tramite": "abierto",
    "en trámite": "abierto",
    "rechazado": "rechazado",
    "denegado": "rechazado",
}
siniestros["estado"] = siniestros["estado"].astype(str).str.strip().str.lower().map(mapa_estado_siniestro)
assert siniestros["estado"].isna().sum() == 0, "Quedaron nulos en siniestros.estado"

valores_fraude_true = {"si", "sí", "s�", "1", "true"}
siniestros["fraude"] = (
    siniestros["fraude"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(valores_fraude_true)
)

# 6) Limpieza ligera de clientes y mediadores
mapa_provincia = {
    "mad": "Madrid", "madrid": "Madrid",
    "bcn": "Barcelona", "barcelona": "Barcelona",
    "vlc": "Valencia", "valencia": "Valencia",
    "sev": "Sevilla", "sevilla": "Sevilla",
    "mlg": "Málaga", "malaga": "Málaga", "málaga": "Málaga", "m�laga": "Málaga",
    "zgz": "Zaragoza", "zaragoza": "Zaragoza",
    "bil": "Bilbao", "bilbao": "Bilbao", "bizkaia": "Bilbao",
    "ali": "Alicante", "alicante": "Alicante",
}
prov_norm = clientes["provincia"].fillna("desconocido").astype(str).str.strip().str.lower()
clientes["provincia"] = prov_norm.map(lambda x: mapa_provincia.get(x, x.capitalize() if x != "desconocido" else "desconocido"))

mapa_sexo = {"mujer": "mujer", "f": "mujer", "femenino": "mujer", "hombre": "hombre", "m": "hombre", "masculino": "hombre"}
clientes["sexo"] = clientes["sexo"].fillna("desconocido").astype(str).str.strip().str.lower().map(lambda x: mapa_sexo.get(x, "desconocido"))

mediadores["zona"] = mediadores["zona"].astype(str).str.strip().str.lower()
mapa_tipo_med = {"corredor": "corredor", "agente": "agente", "agencia": "agencia"}
mediadores["tipo"] = mediadores["tipo"].astype(str).str.strip().str.lower().map(mapa_tipo_med)

print("Normalización completada con 0 nulos nuevos en canal, ramo, tipo, estado y fraude.")
display(productos)


Normalización completada con 0 nulos nuevos en canal, ramo, tipo, estado y fraude.


,producto_id,nombre,ramo,franquicia,ramo_agrupado
0,1,auto,auto,300.0,auto
1,2,hogar,hogar,150.0,patrimonial
2,3,salud,salud,0.0,personal
3,4,vida,vida,NaN,personal
4,5,mascotas,mascotas,90.0,patrimonial


### 3. Deduplicación y Filtrado de Valores Imposibles / Outliers
Aplicamos las reglas de negocio identificadas con Hugo:
- **Pólizas**: deduplicar por `poliza_id` (`keep='first'`) y filtrar `prima_anual > 0`.
- **Siniestros**: deduplicar por `siniestro_id` (`keep='first'`) y filtrar `importe > 0` e `importe < 1_000_000` (descartando recobros negativos y el código centinela `9.999.999 €`), **conservando los siniestros abiertos** (`fecha_cierre` vacía).


In [4]:
# Pólizas: deduplicar y filtrar prima > 0
filas_pol_antes = len(polizas)
polizas = polizas.drop_duplicates(subset=["poliza_id"], keep="first").copy()
filas_pol_dedup = len(polizas)

primas_invalidas = polizas[polizas["prima_anual"].isna() | (polizas["prima_anual"] <= 0)]
print(f"Pólizas con prima <= 0 o nula a excluir: {len(primas_invalidas)}")
display(primas_invalidas[["poliza_id", "canal", "prima_anual", "tipo", "estado"]])

polizas = polizas[polizas["prima_anual"] > 0].copy()
print(f"Pólizas: {filas_pol_antes:,} crudas -> {filas_pol_dedup:,} sin duplicados (-{filas_pol_antes - filas_pol_dedup}) -> {len(polizas):,} con prima > 0 (-{len(primas_invalidas)})")
print("Desglose por tipo de póliza limpia:")
print(polizas["tipo"].value_counts())

# Siniestros: deduplicar y filtrar 0 < importe < 1.000.000
filas_sin_antes = len(siniestros)
siniestros = siniestros.drop_duplicates(subset=["siniestro_id"], keep="first").copy()
filas_sin_dedup = len(siniestros)

outliers_sin = siniestros[siniestros["importe"].isna() | (siniestros["importe"] <= 0) | (siniestros["importe"] >= 1_000_000)]
print(f"\nSiniestros con importe <= 0 o >= 1.000.000 a excluir: {len(outliers_sin)}")
display(outliers_sin[["siniestro_id", "poliza_id", "importe", "estado", "fecha_cierre"]])

siniestros = siniestros[(siniestros["importe"] > 0) & (siniestros["importe"] < 1_000_000)].copy()
print(f"Siniestros: {filas_sin_antes:,} crudos -> {filas_sin_dedup:,} sin duplicados (-{filas_sin_antes - filas_sin_dedup}) -> {len(siniestros):,} con importe válido (-{len(outliers_sin)})")
print(f"Siniestros abiertos conservados (fecha_cierre NaT): {siniestros['fecha_cierre'].isna().sum():,}")


Pólizas con prima <= 0 o nula a excluir: 15


,poliza_id,canal,prima_anual,tipo,estado
25863,835412,banca,-120.0,nueva,vigente
26710,835405,banca,NaN,nueva,anulada
38931,835410,comparadores,NaN,nueva,vigente
39723,835401,comparadores,0.0,nueva,vigente
42358,835408,comparadores,0.0,nueva,vigente
48515,835414,mediadores,-450.0,nueva,vigente
58634,835413,mediadores,0.0,nueva,vigente
63280,835406,referido,0.0,nueva,anulada
63375,835415,comparadores,NaN,nueva,vigente
70785,835403,comparadores,0.0,nueva,vigente


Pólizas: 96,015 crudas -> 95,415 sin duplicados (-600) -> 95,400 con prima > 0 (-15)
Desglose por tipo de póliza limpia:
tipo
nueva         60000
renovacion    35400
Name: count, dtype: int64

Siniestros con importe <= 0 o >= 1.000.000 a excluir: 30


,siniestro_id,poliza_id,importe,estado,fecha_cierre
3206,622501,147689,-500.0,cerrado,2024-01-31
11024,622507,115512,9999999.0,cerrado,2023-02-04
11544,622486,158025,-1250.0,cerrado,2023-05-17
14591,622505,127066,-500.0,cerrado,2024-08-29
16958,622480,141709,9999999.0,abierto,NaT
18026,622492,100741,9999999.0,abierto,NaT
18661,622499,101274,9999999.0,cerrado,2023-07-18
22524,622482,131883,-1250.0,cerrado,2023-04-16
22637,622496,155592,9999999.0,cerrado,2022-10-19
32475,622500,141029,9999999.0,cerrado,2023-06-27


Siniestros: 123,366 crudos -> 122,509 sin duplicados (-857) -> 122,479 con importe válido (-30)
Siniestros abiertos conservados (fecha_cierre NaT): 20,417


### 4. Exportación de las 7 Tablas Limpias a `data/clean/` y Foto de Superficie Inicial
Guardamos las tablas saneadas para trabajar con SQL a partir de mañana y verificamos el recuento de las **60.000 pólizas nuevas** por canal y de los costes de campaña por canal.


In [5]:
# Guardamos las 7 tablas limpias
canales.to_csv(CLEAN_DIR / "canales.csv", index=False)
productos.to_csv(CLEAN_DIR / "productos.csv", index=False)
mediadores.to_csv(CLEAN_DIR / "mediadores.csv", index=False)
campanas.to_csv(CLEAN_DIR / "campanas.csv", index=False)
clientes.to_csv(CLEAN_DIR / "clientes.csv", index=False)
polizas.to_csv(CLEAN_DIR / "polizas.csv", index=False)
siniestros.to_csv(CLEAN_DIR / "siniestros.csv", index=False)

polizas_nuevas = polizas[polizas["tipo"] == "nueva"]
resumen_inicial = (
    polizas_nuevas.groupby("canal", as_index=False)
    .agg(
        polizas_nuevas=("poliza_id", "count"),
        prima_total=("prima_anual", "sum"),
        prima_media=("prima_anual", "mean"),
    )
    .merge(
        campanas.groupby("canal", as_index=False).agg(coste_campanas=("coste", "sum")),
        on="canal",
        how="left",
    )
)
resumen_inicial["cuota_polizas_pct"] = (resumen_inicial["polizas_nuevas"] / resumen_inicial["polizas_nuevas"].sum() * 100).round(2)
resumen_inicial["cac_eur"] = (resumen_inicial["coste_campanas"] / resumen_inicial["polizas_nuevas"]).round(2)
resumen_inicial = resumen_inicial.sort_values("polizas_nuevas", ascending=False)
display(resumen_inicial)


,canal,polizas_nuevas,prima_total,prima_media,coste_campanas,cuota_polizas_pct,cac_eur
1,comparadores,24000,11520000.07,480.000003,360000.0,40.0,15.0
2,directo,13200,5543999.73,419.999980,396000.0,22.0,30.0
3,mediadores,10800,6587999.95,609.999995,972000.0,18.0,90.0
0,banca,7200,2807999.68,389.999956,324000.0,12.0,45.0
4,referido,4800,2159999.95,449.999990,96000.0,8.0,20.0


### 5. La Decisión del Día: ¿Qué es el «Margen» de un Canal?

Antes de abrir mañana las consultas en SQL, dejamos escrita la definición defendible ante Marta (CEO) y Rubén (Lead de Analítica y Actuarial):

> **Definición operativa:** El margen de un canal es el **margen técnico medio de sus pólizas nuevas** —lo que sobra de la prima tras pagar siniestros, gastos de gestión y coste de captación—, respaldado por su *loss ratio*, su *combined ratio* y su *tasa de renovación*; y su aportación al negocio es el **margen técnico total** que suma o resta al fondo común de Ampara.

En fórmulas:
1. **Loss Ratio del canal**:
   $$\text{Loss Ratio} = \frac{\sum \text{Importe de siniestros (incluyendo abiertos)}}{\sum \text{Prima anual suscrita}}$$
2. **Combined Ratio del canal** (con gastos de gestión fijos del 22% en Ampara):
   $$\text{Combined Ratio} = \text{Loss Ratio} + 0,22 + \frac{\text{CAC}}{\text{Prima media}}$$
   *(Si $\text{Combined Ratio} > 100%$, el canal pierde dinero suscribiendo).*
3. **Margen Técnico por Póliza y Margen Técnico Total**:
   $$\text{Margen por póliza} = \text{Prima media} \times (1 - \text{Combined Ratio})$$
   $$\text{Margen total del canal} = \text{Margen por póliza} \times N_{\text{pólizas nuevas}}$$
